# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:

- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

The agent handles:
- **Math queries** → Calculator Tool
- **Keyword extraction** → Keyword Tool
- **General queries** → Direct response

### What's implemented
- Agent logic with conditional (regex-based) routing
- Tool integration (Calculator, Keyword Extractor, + a bonus Text Stats tool)
- Basic + robust error handling
- Structured JSON output for every response
- **Bonus:** improved routing (regex, not just substring match), logging, and an extra tool


## 🔧 Setup & Logging

We set up a simple logger so every routing decision and tool call is traceable — useful for debugging and for demonstrating the agent's reasoning trail.

In [ ]:
import re
import json
import logging
from datetime import datetime

# --- Logging setup -----------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("agent")


## 🛠️ TOOL 1: Calculator

Evaluates a math expression. Instead of a raw `eval()` (unsafe — it would execute arbitrary Python), we validate the expression only contains numeric/operator characters before evaluating.

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        expression = expression.strip()
        if not expression:
            return "Error: empty expression"

        # Only allow digits, operators, parentheses, decimal points, and spaces
        if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):
            return "Error: expression contains invalid characters"

        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except ZeroDivisionError:
        return "Error: division by zero"
    except Exception as e:
        return f"Error in calculation: {e}"


# quick sanity check
print(calculator("20 + 5"))
print(calculator("10 / 0"))
print(calculator("import os"))  # should be rejected


## 🛠️ TOOL 2: Keyword Extractor

Pulls out likely-important words (length > 4, deduplicated, lower-cased) from a piece of text.

In [ ]:
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        text = text.strip()
        if not text:
            return []
        words = re.findall(r"[A-Za-z]+", text)
        keywords = sorted(set(w.lower() for w in words if len(w) > 4))
        return keywords[:5]
    except Exception as e:
        logger.error(f"Keyword extraction failed: {e}")
        return []


print(extract_keywords("Artificial Intelligence is transforming industries"))


## 🛠️ TOOL 3 (Bonus): Text Stats Tool

An extra tool the agent can route to — returns basic statistics about a piece of text (word count, character count, sentence count). Demonstrates that the pipeline scales to more tools without changing its core design.

In [ ]:
def text_stats(text: str) -> dict:
    """Return basic statistics about a piece of text."""
    try:
        text = text.strip()
        words = re.findall(r"\w+", text)
        sentences = re.split(r"[.!?]+", text)
        sentences = [s for s in sentences if s.strip()]
        return {
            "word_count": len(words),
            "char_count": len(text),
            "sentence_count": len(sentences),
        }
    except Exception as e:
        logger.error(f"text_stats failed: {e}")
        return {"error": str(e)}


print(text_stats("Agentic AI pipelines are fun. They route tasks intelligently!"))


## 🤖 Agent Logic

**Routing strategy (bonus: improved routing):**

Rather than a plain `"calculate" in query_lower` substring check (which would
misfire on things like *"let's calculate our next steps"* with no numbers),
the router uses regex patterns to:

1. **Math intent** — trigger words (`calculate`, `compute`, `evaluate`, `what is`,
   `solve`) *or* the presence of an actual arithmetic expression (digits + operators)
   in the query. The arithmetic expression is extracted out of the sentence before
   being sent to the calculator tool.
2. **Keyword intent** — trigger words (`keyword`, `keywords`, `extract`) route to
   the keyword tool, using the text *after* a phrase like "from ..." if present,
   otherwise the whole query.
3. **Stats intent** — trigger words (`stats`, `statistics`, `word count`, `count words`).
4. **General / fallback** — anything else gets a direct rule-based response from a
   tiny built-in knowledge base, with a graceful fallback message otherwise.

Every branch is wrapped in error handling, logs its decision, and returns a
**consistent structured JSON shape**.


In [ ]:
# --- Simple built-in knowledge base for general queries ----------------
GENERAL_RESPONSES = {
    "machine learning": "Machine learning is a field of AI where systems learn patterns from data instead of being explicitly programmed.",
    "artificial intelligence": "Artificial Intelligence (AI) is the broader field of building systems that can perform tasks requiring human-like intelligence.",
    "deep learning": "Deep learning uses multi-layered neural networks to learn complex patterns directly from raw data.",
    "agentic ai": "Agentic AI refers to systems that can autonomously plan, decide, and use tools to accomplish a goal, rather than just responding once.",
}

MATH_TRIGGER_RE = re.compile(r"\b(calculate|compute|evaluate|solve)\b", re.IGNORECASE)
MATH_EXPR_RE = re.compile(r"[-+]?\d[\d\s+\-*/().%]*\d|\b\d+\b")
KEYWORD_TRIGGER_RE = re.compile(r"\b(keywords?|extract)\b", re.IGNORECASE)
KEYWORD_FROM_RE = re.compile(r"\bfrom\b\s*(.*)", re.IGNORECASE)
STATS_TRIGGER_RE = re.compile(r"\b(stats|statistics|word count|count words)\b", re.IGNORECASE)


def agent(query: str) -> dict:
    """Route a user query to the right tool and return structured JSON."""
    timestamp = datetime.now().isoformat(timespec="seconds")

    if not isinstance(query, str) or not query.strip():
        logger.warning("Received empty/invalid query")
        return {
            "type": "error",
            "result": "Query must be a non-empty string",
            "query": query,
            "timestamp": timestamp,
        }

    query_clean = query.strip()
    query_lower = query_clean.lower()

    try:
        # ---- 1. Math routing -------------------------------------------------
        if MATH_TRIGGER_RE.search(query_lower) or MATH_EXPR_RE.search(query_clean):
            expr_match = MATH_EXPR_RE.search(query_clean)
            if expr_match:
                logger.info(f"Routing to calculator | query='{query_clean}'")
                result = calculator(expr_match.group())
                status = "error" if result.startswith("Error") else "success"
                return {
                    "type": "calculation",
                    "result": result,
                    "query": query_clean,
                    "status": status,
                    "timestamp": timestamp,
                }

        # ---- 2. Keyword routing ------------------------------------------
        if KEYWORD_TRIGGER_RE.search(query_lower):
            from_match = KEYWORD_FROM_RE.search(query_clean)
            text_for_keywords = from_match.group(1) if from_match else query_clean
            logger.info(f"Routing to keyword extractor | text='{text_for_keywords}'")
            result = extract_keywords(text_for_keywords)
            return {
                "type": "keywords",
                "result": result,
                "query": query_clean,
                "status": "success" if result else "no_keywords_found",
                "timestamp": timestamp,
            }

        # ---- 3. Stats routing (bonus tool) --------------------------------
        if STATS_TRIGGER_RE.search(query_lower):
            logger.info(f"Routing to text_stats | query='{query_clean}'")
            result = text_stats(query_clean)
            return {
                "type": "stats",
                "result": result,
                "query": query_clean,
                "status": "success",
                "timestamp": timestamp,
            }

        # ---- 4. General / fallback -----------------------------------------
        logger.info(f"Routing to general response | query='{query_clean}'")
        for key, answer in GENERAL_RESPONSES.items():
            if key in query_lower:
                return {
                    "type": "general",
                    "result": answer,
                    "query": query_clean,
                    "status": "success",
                    "timestamp": timestamp,
                }

        return {
            "type": "general",
            "result": "I don't have a specific answer for that yet, but I've logged it as a general query.",
            "query": query_clean,
            "status": "fallback",
            "timestamp": timestamp,
        }

    except Exception as e:
        logger.error(f"Agent failed on query '{query_clean}': {e}")
        return {
            "type": "error",
            "result": f"Unexpected error: {e}",
            "query": query_clean,
            "status": "error",
            "timestamp": timestamp,
        }


## 📦 Expected Output Format

```json
{
  "type": "calculation / keywords / stats / general / error",
  "result": ...,
  "query": "...",
  "status": "success / error / fallback / no_keywords_found",
  "timestamp": "..."
}
```

## 🧪 Test Cases

In [ ]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Give me stats on: Agentic AI pipelines are fun to build.",
    "10 / 0",
    "",
]

for q in queries:
    print("Query:", repr(q))
    response = agent(q)
    print("Response:", json.dumps(response, indent=2))
    print("-" * 60)


## 🎯 Interactive Mode

Run this cell in Colab/Jupyter to chat with the agent live. Type `exit` to stop.

In [ ]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        print("Session ended.")
        break
    response = agent(user_input)
    print("Response:", json.dumps(response, indent=2))
